# Prototipo del agente ganadero (LangChain + Gemini)

Este notebook valida localmente el funcionamiento del agente antes del deploy en OCI:

1. Carga y exploracion del dataset ganadero
2. Construccion del agente
3. Bateria de 10 preguntas de prueba en lenguaje natural

**Requisito previo:** tener un archivo `.env` en la raiz del proyecto con `GOOGLE_API_KEY` configurada (ver `.env.example`).

In [1]:
# Agregar la raiz del proyecto al path para importar src desde notebooks/
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raíz del proyecto: {PROJECT_ROOT}")

Raíz del proyecto: D:\Proyectos personales\Alura agente\Challenge-Alura-Agente


## 1. Carga del dataset

In [2]:
from src.data_loader import load_dataset, validate_dataset, get_dataset_summary

df = load_dataset(str(PROJECT_ROOT / "data" / "ganaderia_dataset.csv"))
validate_dataset(df)

print(f"Dataset cargado y validado: {len(df)} registros")
df.head()

Dataset cargado y validado: 835 registros


,id_animal,codani,finca,raza,sexo,fecha_nacimiento,peso_nacimiento,fecha_destete,peso_destete,id_madre,id_toro,estado,fecha_evento,costo_alimentacion_mensual
0,CR-00001,9707587591,La Libertad,Nelore,M,2024-02-14,30.0,2024-10-09,132.9,NaN,TORO-05,Activo,NaT,35255.0
1,CR-00002,4334137354,Nispero,Simmental,M,2024-09-03,33.6,2025-04-06,174.4,CR-00651,TORO-12,Activo,NaT,29011.0
2,CR-00003,4636309516,La Libertad,Nelore,H,2023-08-09,22.0,2024-03-22,172.9,NaN,TORO-07,Activo,NaT,24694.0
3,CR-00004,4415176717,La Libertad,Brahman,H,2025-01-22,31.8,2025-09-09,227.2,CR-00129,TORO-12,Activo,NaT,23603.0
4,CR-00005,8680325134,Miravalles,Angus,H,2024-03-02,30.7,2024-11-24,206.7,CR-00151,TORO-02,Activo,NaT,19081.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id_animal                   835 non-null    object        
 1   codani                      835 non-null    object        
 2   finca                       835 non-null    category      
 3   raza                        835 non-null    category      
 4   sexo                        835 non-null    category      
 5   fecha_nacimiento            835 non-null    datetime64[ns]
 6   peso_nacimiento             835 non-null    float64       
 7   fecha_destete               684 non-null    datetime64[ns]
 8   peso_destete                684 non-null    float64       
 9   id_madre                    665 non-null    object        
 10  id_toro                     835 non-null    object        
 11  estado                      835 non-null    category      

In [4]:
# Resumen que se inyecta en el system prompt del agente
print(get_dataset_summary(df))

El dataset contiene 835 registros y 14 columnas.

Columnas y tipos:
  - id_animal: object
  - codani: object
  - finca: category
  - raza: category
  - sexo: category
  - fecha_nacimiento: datetime64[ns]
  - peso_nacimiento: float64
  - fecha_destete: datetime64[ns]
  - peso_destete: float64
  - id_madre: object
  - id_toro: object
  - estado: category
  - fecha_evento: datetime64[ns]
  - costo_alimentacion_mensual: float64

Rangos de fechas:
  - fecha_nacimiento: 2022-01-02 a 2025-12-30
  - fecha_destete: 2022-08-14 a 2026-06-27
  - fecha_evento: 2022-03-08 a 2026-06-30

Rangos numericos (min / promedio / max):
  - peso_nacimiento: 22.0 / 31.9 / 44.4
  - peso_destete: 120.0 / 182.1 / 260.0
  - costo_alimentacion_mensual: 15028.0 / 29330.5 / 44991.0

Valores de las columnas categoricas:
  - finca: ECOS, La Libertad, Las Flores, Miravalles, Nispero
  - raza: Angus, Brahman, Cruzado, Jersey, Nelore, Simmental
  - sexo: H, M
  - estado: Activo, Muerto, Traslado, Vendido

Columnas con valo

## 2. Construccion del agente

In [5]:
from src.agent import build_agent, ask

agent = build_agent(df)
print("Agente construido correctamente")

Agente construido correctamente


## 3. Bateria de preguntas de prueba

Se ejecutan 10 preguntas en lenguaje natural que cubren conteos, promedios,
agrupaciones, filtros y rankings sobre el dataset.

In [6]:
QUESTIONS = [
    "¿Cuántos animales hay registrados en total?",
    "¿Cuál es el peso promedio al destete por finca?",
    "¿Qué finca tuvo más nacimientos en 2024?",
    "¿Cuál es la raza más común en la finca Miravalles?",
    "¿Cuántos animales están en estado Muerto?",
    "¿Cuál toro tiene la mayor cantidad de crías registradas?",
    "¿Cuál es la diferencia de peso al destete entre machos y hembras?",
    "¿Cuántos animales no tienen registro de destete?",
    "¿Cuál es el costo promedio de alimentación mensual en La Libertad?",
    "Muéstrame los 5 animales con mayor peso al destete",
]

print(f"Total de preguntas de prueba: {len(QUESTIONS)}")

Total de preguntas de prueba: 10


In [7]:
# Ejecutar la bateria completa mostrando pregunta y respuesta
# La pausa entre preguntas respeta el limite de solicitudes por minuto
# del tier gratuito de la API de Gemini
import time

for i, question in enumerate(QUESTIONS, start=1):
    print("=" * 70)
    print(f"Pregunta {i}: {question}")
    print("-" * 70)
    answer = ask(agent, question)
    print(f"Respuesta: {answer}")
    print()
    if i < len(QUESTIONS):
        time.sleep(20)

Pregunta 1: ¿Cuántos animales hay registrados en total?
----------------------------------------------------------------------


Respuesta: En total, hay 835 animales registrados en el sistema.



Pregunta 2: ¿Cuál es el peso promedio al destete por finca?
----------------------------------------------------------------------


<agente>:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Respuesta: El peso promedio al destete por finca es el siguiente:

| Finca | Peso Promedio al Destete |
| :--- | :--- |
| ECOS | 179.6 kg |
| La Libertad | 173.9 kg |
| Las Flores | 175.9 kg |
| Miravalles | 208.4 kg |
| Nispero | 173.9 kg |



...

Respuesta: Actualmente, hay 151 animales que no tienen registro de destete en el sistema.



Pregunta 9: ¿Cuál es el costo promedio de alimentación mensual en La Libertad?
----------------------------------------------------------------------


Respuesta: El costo promedio de alimentación mensual en la finca La Libertad es de ₡28 548.3.



Pregunta 10: Muéstrame los 5 animales con mayor peso al destete
----------------------------------------------------------------------


Respuesta: Aquí tienes los 5 animales con el mayor peso al destete registrado:

| Código del Animal | Peso al Destete |
| :--- | :--- |
| 6920769565 | 260.0 kg |
| 7386204565 | 260.0 kg |
| 1789381107 | 260.0 kg |
| 4765600397 | 260.0 kg |
| 2169293548 | 260.0 kg |



## Conclusiones

Si todas las preguntas anteriores devolvieron respuestas coherentes con los datos,
el agente esta listo para integrarse a la interfaz Streamlit (`app.py`) y
posteriormente desplegarse en la VM de Oracle Cloud.

Puntos a verificar manualmente:

- Las respuestas estan en español y con unidades (kg, colones CRC)
- Los conteos coinciden con el resumen del dataset
- Miravalles deberia destacar en peso al destete (variacion intencional)
- 2024 deberia ser el año con mas nacimientos (variacion intencional)